In [7]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import math
import random
import shutil
import time
import csv
import gc
from pathlib import Path
from PIL import Image
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler
from torchvision import transforms

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from tqdm import trange, tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    def trange(n, desc=None): return range(n)
    def tqdm(x, **kw): return x


# ============================================================
# CONFIGURATION
# ============================================================

DEBUG_MODE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CUDA_AVAILABLE = torch.cuda.is_available()

if not CUDA_AVAILABLE:
    torch.set_num_threads(4)

ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()

DATA_DIR = ROOT / "data"
IMAGE_DIR = ROOT / "image"
TRAIN_DIR = IMAGE_DIR / "train"
VAL_DIR = IMAGE_DIR / "validation"
TEST_DIR = IMAGE_DIR / "test"
MODEL_DIR = ROOT / "models"
OUT_DIR = ROOT / "outputs"
FIGURES_DIR = OUT_DIR / "figures"

SCORE_MODEL_PATH = MODEL_DIR / "score_large.pth"
LORA_MODEL_PATH = MODEL_DIR / "lora_v2.pth"

IMG_SIZE = 256
CLAMP_MIN = 0.0
CLAMP_MAX = 1.0

CT_BACKEND = "astra"
NUM_ANGLES = 180
ANGLE_MAX_DEG = 180.0

I0 = 1.5e5
ELECTRONIC_NOISE_STD = 4.0
MU_MAX = 0.5

T = 1000
BETA_START = 1e-4
BETA_END = 0.02

USE_ADMM = True

if DEBUG_MODE:
    ADMM_ITERS = 8
    DATA_PROX_ITERS = 8
    FP_ITERS = 3
    N_TEST_IMAGES = 3
    UQ_SAMPLES = 4
    N_EXAMPLES_TO_SAVE = 2
    UQ_CALIB_N_VAL = 2
    LORA_EPOCHS = 2
    LORA_MAX_IMAGES = 500
    LORA_STEPS_PER_EPOCH = 50
    GRID_N_VAL = 3
else:
    ADMM_ITERS = 35
    DATA_PROX_ITERS = 20
    FP_ITERS = 20                 # V2.7 : 20 iterations (instead of 15)
    N_TEST_IMAGES = 30
    UQ_SAMPLES = 10
    N_EXAMPLES_TO_SAVE = 5
    UQ_CALIB_N_VAL = 5
    LORA_EPOCHS = 10
    LORA_MAX_IMAGES = 8000
    LORA_STEPS_PER_EPOCH = 500
    GRID_N_VAL = 50

# ADMM/HQS
RHO_START = 80.0
RHO_END = 3500.0
DENOISE_T_START = 120
DENOISE_T_END = 10
PRIOR_STRENGTH = 0.25
ADMM_RELAX = 0.75
POWER_ITERS = 20

# ---- FIXED POINT V2.7 (ANCHORED GRADIENT + MOMENTUM + TRUST REGION) ----
FP_ANCHOR_BETA = 0.30                  # anchoring to x_lora (0.30 for a bit more freedom)
FP_STEP_INIT = 0.25                    # initial step (fraction of 1/L)
FP_BACKTRACK_FACTOR = 0.5              # backtracking factor
FP_MAX_BACKTRACK = 3
FP_MIN_REL_IMPROVE = 1e-5              # minimum relative improvement
FP_TRUST_RADIUS = 0.03                 # max L∞ radius around x_lora
FP_USE_MOMENTUM = True                 # Nesterov momentum
FP_MOMENTUM = 0.5                      # momentum coefficient

# LoRA V2.1
LORA_RANK = 16
LORA_ALPHA = 32
LORA_TARGETS = ("c1", "c2", "c3", "c4", "c5", "out")
LORA_BATCH = 16
LORA_LR = 5e-5
LORA_VAL_MAX = 150
LORA_VAL_STEPS = 20
LORA_EARLY_STOP_PATIENCE = 3

FORCE_LORA_RETRAIN = False

# UQ
UQ_NOISE_SCALE = 0.08
UQ_MIN_STD = 1e-4
UQ_ADMM_ITERS = 4
UQ_DATA_PROX_ITERS = 6
UQ_FP_ITERS = 3
UQ_CALIB_ALPHA = 0.05
UQ_MIN_CF = 1.0
UQ_MAX_CF = 50.0
UQ_CALIB_PIXELS_PER_IMAGE = 30000

USE_WEIGHTED_DATA_FIDELITY = True
WEIGHT_MIN = 0.10
WEIGHT_MAX = 10.0

POST_BILATERAL = False

BATCH_LOG_EVERY = 5

SPLIT_TRAIN = 0.80
SPLIT_VAL = 0.10
SPLIT_TEST = 0.10
FORCE_RESPLIT = False

ROI_DEFINITIONS = {
    "center": (0.30, 0.70, 0.30, 0.70),
    "outline": (0.05, 0.95, 0.05, 0.95),
    "top_right": (0.10, 0.40, 0.60, 0.90),
    "bottom_left": (0.60, 0.90, 0.10, 0.40),
}

SEED = 1234
ADJOINT_TOL = 1e-2


# ============================================================
# UTILS
# ============================================================

def ensure_dirs():
    for d in [DATA_DIR, IMAGE_DIR, TRAIN_DIR, VAL_DIR, TEST_DIR,
              MODEL_DIR, OUT_DIR, FIGURES_DIR]:
        d.mkdir(parents=True, exist_ok=True)


def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if CUDA_AVAILABLE:
        torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True


def to_numpy_img(x):
    """Convert to numpy image in [0,1] (clipped)."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().float().squeeze().numpy()
    return np.clip(np.asarray(x, np.float32), 0.0, 1.0)


def to_numpy_raw(x):
    """Convert to numpy float32 WITHOUT clipping (for sinograms)."""
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().float().squeeze().numpy()
    return np.asarray(x, np.float32)


def free_mem():
    gc.collect()
    if CUDA_AVAILABLE:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass


def detect_ct_backend(preferred="auto"):
    if preferred == "grid_sample":
        return "grid_sample"
    try:
        import astra
        if CUDA_AVAILABLE:
            try:
                vg = astra.create_vol_geom(32, 32)
                pg = astra.create_proj_geom(
                    "parallel", 1.0, 32,
                    np.linspace(0, np.pi, 16, endpoint=False, dtype=np.float32))
                pid = astra.create_projector("cuda", pg, vg)
                astra.projector.delete(pid)
                return "astra"
            except Exception:
                return "astra_cpu"
        return "astra_cpu"
    except ImportError:
        return "grid_sample"


# ============================================================
# DATASET
# ============================================================

def prepare_split(seed=1234, force=False):
    exts = ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"]
    all_files = []
    for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
        if folder.exists():
            for ext in exts:
                all_files += list(folder.glob(ext))
    all_files = sorted(all_files)
    n = len(all_files)
    if n == 0:
        raise RuntimeError(f"No image found in {IMAGE_DIR}")

    def _count_images(d):
        return len([f for f in d.glob("*.*")
                    if f.suffix.lower() in (".png", ".jpg", ".jpeg", ".tif", ".tiff")])

    nt = _count_images(TRAIN_DIR)
    nv = _count_images(VAL_DIR)
    ne = _count_images(TEST_DIR)

    if not force and abs(nt / max(n, 1) - SPLIT_TRAIN) < 0.02:
        print(f"[OK] Existing split validated: {nt}|{nv}|{ne}")
        return

    rng = random.Random(seed)
    rng.shuffle(all_files)
    ntr = max(1, int(n * SPLIT_TRAIN))
    nva = max(1, int(n * SPLIT_VAL))
    split = {
        TRAIN_DIR: all_files[:ntr],
        VAL_DIR: all_files[ntr:ntr + nva],
        TEST_DIR: all_files[ntr + nva:],
    }
    for d in split:
        d.mkdir(parents=True, exist_ok=True)
    for td, files in split.items():
        for f in files:
            dst = td / f.name
            if f.resolve() == dst.resolve():
                continue
            try:
                if dst.exists():
                    dst.unlink()
                shutil.move(str(f), str(dst))
            except Exception:
                pass
    print("[OK] New split performed successfully")


class CTImageDataset(Dataset):
    def __init__(self, folder, augment=False, max_images=None, seed=1234):
        self.files = []
        for ext in ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"]:
            self.files += list(Path(folder).glob(ext))
        self.files = sorted(self.files)
        if not self.files:
            raise RuntimeError(f"Empty folder: {folder}")
        if max_images and len(self.files) > max_images:
            rng = random.Random(seed)
            self.files = sorted(rng.sample(self.files, max_images))
        tfs = [
            transforms.Grayscale(1),
            transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
        ]
        if augment:
            tfs += [
                transforms.RandomHorizontalFlip(0.5),
                transforms.RandomRotation(5),
            ]
        tfs += [transforms.ToTensor()]
        self.transform = transforms.Compose(tfs)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        img = Image.open(self.files[i])
        return self.transform(img).clamp(0.0, 1.0)


# ============================================================
# CT SIMULATION
# ============================================================

@torch.no_grad()
def simulate_ct_measurement(y_clean, i0=I0, elec_std=ELECTRONIC_NOISE_STD,
                             mu_max=MU_MAX, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    y_max = max(float(y_clean.max().item()), 1e-8)
    att = (y_clean.clamp(min=0.0) / y_max) * mu_max
    exp_p = i0 * torch.exp(-att)
    pois = torch.sqrt(exp_p.clamp(min=1e-6)) * torch.randn_like(exp_p)
    det = (exp_p + pois + elec_std * torch.randn_like(exp_p)).clamp(min=1.0)
    y_noisy = (-torch.log(det / i0) * y_max / mu_max).clamp(0.0, y_max * 1.5)
    return y_noisy, det


def estimate_data_weights_from_counts(det, elec_std=ELECTRONIC_NOISE_STD):
    if (not USE_WEIGHTED_DATA_FIDELITY) or det is None:
        return None
    if isinstance(det, np.ndarray):
        d = torch.from_numpy(det).float()
    else:
        d = det.detach().float()
    d = d.clamp_min(1.0)
    inv_var = (d * d) / (d + float(elec_std) ** 2 + 1e-12)
    if inv_var.ndim >= 3:
        inv_var = inv_var / (inv_var.mean(dim=(-2, -1), keepdim=True) + 1e-12)
    else:
        inv_var = inv_var / (inv_var.mean() + 1e-12)
    return inv_var.clamp(WEIGHT_MIN, WEIGHT_MAX)


def format_weights(weights, A=None, y=None):
    if (not USE_WEIGHTED_DATA_FIDELITY) or weights is None:
        return None
    if isinstance(weights, np.ndarray):
        w = torch.from_numpy(weights).float()
    elif isinstance(weights, torch.Tensor):
        w = weights.detach().float()
    else:
        w = torch.tensor(weights, dtype=torch.float32)
    if A is not None:
        dev = A.device
    elif isinstance(y, torch.Tensor):
        dev = y.device
    else:
        dev = torch.device(DEVICE)
    w = w.to(dev).float()
    if w.ndim == 2:
        w = w[None]
    if y is not None:
        yy = y
        if isinstance(yy, np.ndarray):
            yy = torch.from_numpy(yy)
        if yy.ndim == 2:
            target_shape = (1,) + tuple(yy.shape)
        else:
            target_shape = tuple(yy.shape)
        if w.shape[0] == 1 and target_shape[0] > 1:
            w = w.expand(target_shape)
        if tuple(w.shape) != target_shape:
            try:
                w = w.reshape(target_shape)
            except Exception:
                raise ValueError(
                    f"Incompatible weights: w={tuple(w.shape)} vs y={target_shape}")
    return w.clamp(WEIGHT_MIN, WEIGHT_MAX)


def sample_sinogram_for_uq(y, weights=None, scale=UQ_NOISE_SCALE):
    if isinstance(y, np.ndarray):
        yd = torch.from_numpy(y).float().to(DEVICE)
    else:
        yd = y.detach().float()
    ss = max(float(yd.std().item()), 1e-6)
    w = format_weights(weights, A=None, y=yd)
    if w is None:
        yp = yd + scale * ss * torch.randn_like(yd)
    else:
        w = w.to(yd.device).float().clamp(WEIGHT_MIN, WEIGHT_MAX)
        local = torch.rsqrt(w)
        local = local / (local.square().mean().sqrt() + 1e-12)
        sigma = scale * ss * local
        yp = yd + sigma * torch.randn_like(yd)
    return yp.clamp_min(0.0)


# ============================================================
# CT OPERATOR
# ============================================================

class CTOperator:
    def __init__(self, img_size=256, num_angles=180, device="cpu",
                 angle_max_deg=180.0, backend="auto"):
        self.img_size = int(img_size)
        self.num_angles = int(num_angles)
        self.device = torch.device(device)
        self.angle_max_deg = float(angle_max_deg)
        self.backend = detect_ct_backend(backend)
        self._ata_norm = None
        self.angles_deg = np.linspace(0.0, self.angle_max_deg, self.num_angles,
                                       endpoint=False, dtype=np.float32)
        self.angles_rad = np.deg2rad(self.angles_deg).astype(np.float32)
        self.det_count = self.img_size

        try:
            if self.backend in ("astra", "astra_cpu"):
                self._init_astra(use_cuda=(self.backend == "astra"))
            else:
                self._init_grid_sample()
        except Exception as e:
            print(f"[WARN] Fallback grid_sample: {e}")
            self.backend = "grid_sample"
            self._init_grid_sample()

    def _init_astra(self, use_cuda=True):
        import astra
        self.astra = astra
        self.vol_geom = astra.create_vol_geom(self.img_size, self.img_size)
        self.det_count = int(np.ceil(self.img_size * np.sqrt(2)))
        if self.det_count % 2 != 0:
            self.det_count += 1
        self.proj_geom = astra.create_proj_geom("parallel", 1.0,
                                                self.det_count, self.angles_rad)
        ptype = "cuda" if (use_cuda and CUDA_AVAILABLE) else "linear"
        self.proj_id = astra.create_projector(ptype, self.proj_geom, self.vol_geom)
        self.astra_gpu = ptype == "cuda"

    def _astra_forward(self, x_np):
        sid, sino = self.astra.create_sino(x_np, self.proj_id)
        self.astra.data2d.delete(sid)
        return sino.astype(np.float32)

    def _astra_backward(self, sino_np):
        bid, bp = self.astra.create_backprojection(sino_np, self.proj_id)
        self.astra.data2d.delete(bid)
        return bp.astype(np.float32)

    def _astra_fbp(self, sino_np):
        rid = self.astra.data2d.create("-vol", self.vol_geom)
        sid = self.astra.data2d.create("-sino", self.proj_geom, sino_np)
        cfg = (self.astra.astra_dict("FBP_CUDA") if self.astra_gpu
               else self.astra.astra_dict("FBP"))
        if not self.astra_gpu:
            cfg["ProjectorId"] = self.proj_id
        cfg["ReconstructionDataId"] = rid
        cfg["ProjectionDataId"] = sid
        cfg["option"] = {"FilterType": "shepp-logan"}
        aid = self.astra.algorithm.create(cfg)
        try:
            self.astra.algorithm.run(aid)
            rec = self.astra.data2d.get(rid)
        finally:
            self.astra.algorithm.delete(aid)
            self.astra.data2d.delete(rid)
            self.astra.data2d.delete(sid)
        return rec.astype(np.float32)

    def clean_astra_tables(self):
        if hasattr(self, "astra"):
            try:
                self.astra.data2d.clear()
                self.astra.algorithm.clear()
            except Exception:
                pass

    def _init_grid_sample(self):
        self._grid_fwd = [self._make_grid(a) for a in self.angles_deg]
        self._grid_bwd = [self._make_grid(-a) for a in self.angles_deg]

    def _make_grid(self, angle_deg):
        N = self.img_size
        a = math.radians(float(angle_deg))
        c = math.cos(a); s = math.sin(a)
        theta = torch.tensor([[[c, -s, 0.0], [s, c, 0.0]]],
                             dtype=torch.float32, device=self.device)
        return F.affine_grid(theta, (1, 1, N, N), align_corners=False)

    def format_x(self, x):
        if isinstance(x, np.ndarray):
            x = torch.from_numpy(x).float()
        x = x.to(self.device).float()
        if x.ndim == 2:
            x = x[None, None]
        elif x.ndim == 3:
            x = x[:, None]
        return x

    def format_y(self, y):
        if isinstance(y, np.ndarray):
            y = torch.from_numpy(y).float()
        y = y.to(self.device).float()
        if y.ndim == 2:
            y = y[None]
        return y

    def forward(self, x):
        x = self.format_x(x)
        B = x.shape[0]
        if self.backend in ("astra", "astra_cpu"):
            outs = []
            for b in range(B):
                xnp = x[b, 0].detach().cpu().numpy().astype(np.float32)
                outs.append(self._astra_forward(xnp))
            return torch.from_numpy(np.stack(outs)).to(self.device)
        else:
            N = self.img_size
            sc = math.sqrt(N)
            projs = []
            for grid in self._grid_fwd:
                rot = F.grid_sample(x, grid.expand(B, -1, -1, -1),
                                     mode="bilinear", padding_mode="zeros",
                                     align_corners=False)
                projs.append(rot.sum(dim=2).squeeze(1) / sc)
            return torch.stack(projs, dim=1)

    __call__ = forward

    def at(self, y):
        y = self.format_y(y)
        B = y.shape[0]
        if self.backend in ("astra", "astra_cpu"):
            outs = []
            for b in range(B):
                snp = y[b].detach().cpu().numpy().astype(np.float32)
                outs.append(self._astra_backward(snp))
            return torch.from_numpy(np.stack(outs))[:, None].to(self.device)
        else:
            N = self.img_size
            sc = math.sqrt(N)
            back = torch.zeros(B, 1, N, N, device=self.device, dtype=y.dtype)
            for i, grid in enumerate(self._grid_bwd):
                strip = (y[:, i, :][:, None, None, :].expand(B, 1, N, N) / sc)
                back += F.grid_sample(strip, grid.expand(B, -1, -1, -1),
                                       mode="bilinear", padding_mode="zeros",
                                       align_corners=False)
            return back

    @torch.no_grad()
    def fbp(self, y):
        y = self.format_y(y)
        B = y.shape[0]
        N = self.img_size
        if self.backend in ("astra", "astra_cpu"):
            try:
                outs = []
                for b in range(B):
                    snp = y[b].detach().cpu().numpy().astype(np.float32)
                    outs.append(self._astra_fbp(snp))
                rec = torch.from_numpy(np.stack(outs))[:, None].to(self.device)
                return rec.clamp(0.0, 1.0)
            except Exception as e:
                print(f"[WARN] ASTRA FBP error: {e}")
        try:
            from skimage.transform import iradon
            outs = []
            for b in range(B):
                sino = y[b].detach().cpu().numpy().T * math.sqrt(N)
                try:
                    rec = iradon(sino, theta=self.angles_deg, output_size=N,
                                 filter_name="shepp-logan", circle=False)
                except TypeError:
                    rec = iradon(sino, theta=self.angles_deg, output_size=N,
                                 filter="shepp-logan", circle=False)
                outs.append(rec.astype(np.float32))
            return torch.from_numpy(np.stack(outs))[:, None].to(self.device).clamp(0.0, 1.0)
        except Exception:
            return (self.at(y) / max(float(self.num_angles), 1.0)).clamp(0.0, 1.0)

    @torch.no_grad()
    def estimate_ata_norm(self, n_iter=POWER_ITERS):
        if self._ata_norm is not None:
            return self._ata_norm
        N = self.img_size
        x = torch.randn(1, 1, N, N, device=self.device)
        x = x / (x.norm() + 1e-12)
        val = torch.tensor(1.0, device=self.device)
        for _ in range(n_iter):
            AtAx = self.at(self.forward(x))
            val = (x * AtAx).sum() / ((x * x).sum() + 1e-12)
            x = AtAx / (AtAx.norm() + 1e-12)
        self._ata_norm = float(val.abs().item()) + 1e-6
        return self._ata_norm

    def __del__(self):
        if hasattr(self, "astra") and hasattr(self, "proj_id"):
            try:
                self.astra.projector.delete(self.proj_id)
            except Exception:
                pass


# ============================================================
# METRICS
# ============================================================

def compute_psnr(pred, target, dr=1.0):
    p = to_numpy_img(pred); t = to_numpy_img(target)
    mse = float(np.mean((p - t) ** 2))
    return 99.0 if mse <= 1e-12 else 20.0 * math.log10(dr / math.sqrt(mse))


def compute_ssim(pred, target, dr=1.0):
    try:
        from skimage.metrics import structural_similarity
        return float(structural_similarity(to_numpy_img(target), to_numpy_img(pred),
                                            data_range=dr))
    except Exception:
        return 0.0


def compute_all_metrics(pred, target):
    p = to_numpy_img(pred); t = to_numpy_img(target)
    mse = float(np.mean((p - t) ** 2))
    return {
        "psnr": 99.0 if mse < 1e-12 else 20.0 * math.log10(1.0 / math.sqrt(mse)),
        "ssim": compute_ssim(pred, target),
        "rmse": float(np.sqrt(mse)),
        "mae": float(np.mean(np.abs(p - t))),
        "nrmse": float(np.sqrt(mse) / (t.max() - t.min() + 1e-12)),
    }


def compute_roi_metrics(pred, target, roi_defs=None):
    if roi_defs is None:
        roi_defs = ROI_DEFINITIONS
    pn = to_numpy_img(pred); tn = to_numpy_img(target)
    H, W = pn.shape
    r = {}
    for name, (y1, y2, x1, x2) in roi_defs.items():
        pr = pn[int(y1 * H):int(y2 * H), int(x1 * W):int(x2 * W)]
        tr = tn[int(y1 * H):int(y2 * H), int(x1 * W):int(x2 * W)]
        mse = np.mean((pr - tr) ** 2)
        psnr = 99.0 if mse < 1e-12 else 20.0 * math.log10(1.0 / math.sqrt(mse))
        try:
            from skimage.metrics import structural_similarity
            ssim = float(structural_similarity(tr, pr, data_range=1.0))
        except Exception:
            ssim = 0.0
        r[name] = {"psnr": float(psnr), "ssim": ssim,
                   "rmse": float(np.sqrt(mse)), "mae": float(np.mean(np.abs(pr - tr)))}
    return r


# ============================================================
# ADJOINT AND PHANTOM TESTS
# ============================================================

@torch.no_grad()
def test_adjoint(A, tol=ADJOINT_TOL):
    x = torch.randn(1, 1, IMG_SIZE, IMG_SIZE, device=DEVICE)
    y = torch.randn(1, A.num_angles, A.det_count, device=DEVICE)
    Ax = A.forward(x); ATy = A.at(y)
    lhs = torch.sum(Ax * y).item(); rhs = torch.sum(x * ATy).item()
    err = abs(lhs - rhs) / (abs(lhs) + abs(rhs) + 1e-8)
    status = "[OK]" if err < tol else "[FAIL]"
    print(f"\n[ADJOINT] <Ax,y>={lhs:.4e} | <x,ATy>={rhs:.4e} | err={err:.2e} {status}")
    if err >= tol:
        raise RuntimeError(f"Adjoint test failed: err={err:.2e}")
    return err


@torch.no_grad()
def test_phantom(A):
    N = IMG_SIZE
    phantom = torch.zeros(1, 1, N, N, device=DEVICE)
    yy, xx = torch.meshgrid(torch.arange(N, device=DEVICE),
                             torch.arange(N, device=DEVICE), indexing="ij")
    cx = N // 2; cy = N // 2; r = N // 4
    phantom[0, 0, ((xx - cx) ** 2 + (yy - cy) ** 2) < r ** 2] = 1.0
    sino = A.forward(phantom); recon = A.fbp(sino)
    psnr = compute_psnr(recon, phantom)
    print(f"[PHANTOM] Sinogram: {tuple(sino.shape)} | FBP PSNR={psnr:.2f} dB")
    return psnr


# ============================================================
# SCORE MODEL + LORA
# ============================================================

class TimeEmbedding(nn.Module):
    def __init__(self, ts=1000, dim=256):
        super().__init__()
        self.emb = nn.Embedding(ts, dim)

    def forward(self, t):
        return self.emb(t)


class LargeUNet(nn.Module):
    def __init__(self, ts=1000):
        super().__init__()
        self.time = TimeEmbedding(ts, 256)
        self.c1 = nn.Conv2d(1, 64, 3, padding=1)
        self.c2 = nn.Conv2d(64, 128, 3, padding=1)
        self.c3 = nn.Conv2d(128, 256, 3, padding=1)
        self.c4 = nn.Conv2d(256, 128, 3, padding=1)
        self.c5 = nn.Conv2d(128, 64, 3, padding=1)
        self.out = nn.Conv2d(64, 1, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t):
        te = self.time(t)[:, :, None, None]
        h = self.act(self.c1(x))
        h = self.act(self.c2(h))
        h = h + te[:, :128]
        h = self.act(self.c3(h))
        h = self.act(self.c4(h))
        h = self.act(self.c5(h))
        return self.out(h)


def load_score_model(path, device, ts=1000):
    model = LargeUNet(ts).to(device)
    state = torch.load(path, map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    clean = {k.replace("module.", ""): v for k, v in state.items()}
    try:
        model.load_state_dict(clean, strict=True)
    except Exception:
        model.load_state_dict(clean, strict=False)
    model.eval()
    return model


class LoRAConv2d(nn.Module):
    def __init__(self, base, rank=8, alpha=16.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.rank = int(rank)
        self.scale = float(alpha) / max(float(rank), 1.0)
        self.lora_down = nn.Conv2d(base.in_channels, self.rank,
                                    kernel_size=base.kernel_size,
                                    stride=base.stride, padding=base.padding, bias=False)
        self.lora_up = nn.Conv2d(self.rank, base.out_channels, 1, bias=False)
        nn.init.kaiming_uniform_(self.lora_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_up.weight)

    def forward(self, x):
        return self.base(x) + self.scale * self.lora_up(self.lora_down(x))


def inject_lora(model, rank=8, alpha=16.0, targets=LORA_TARGETS):
    for name, child in list(model.named_children()):
        if isinstance(child, nn.Conv2d) and name in targets:
            setattr(model, name, LoRAConv2d(child, rank=rank, alpha=alpha))
        else:
            inject_lora(child, rank=rank, alpha=alpha, targets=targets)
    return model


def load_lora_weights(model, path, device="cpu"):
    state = torch.load(path, map_location=device)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    if isinstance(state, dict) and "lora" in state:
        state = state["lora"]
    clean = {k.replace("module.", ""): v for k, v in state.items() if "lora_" in k}
    model.load_state_dict(clean, strict=False)
    print(f"[OK] LoRA loaded ({len(clean)} tensors)")


def save_lora_weights(model, path):
    lora = {k: v.detach().cpu() for k, v in model.state_dict().items() if "lora_" in k}
    torch.save(lora, path)


_ddpm_beta = torch.linspace(BETA_START, BETA_END, T)
_ddpm_alpha_bar = torch.cumprod(1.0 - _ddpm_beta, dim=0)


def ddpm_noise_image(x, t):
    noise = torch.randn_like(x)
    a = _ddpm_alpha_bar.to(x.device)[t][:, None, None, None]
    xt = torch.sqrt(a) * x + torch.sqrt(1.0 - a) * noise
    return xt, noise


class DiffusionScorePrior:
    def __init__(self, model, device):
        self.model = model.to(device)
        self.model.eval()
        self.device = torch.device(device)
        beta = torch.linspace(BETA_START, BETA_END, T, device=self.device)
        self.alpha_bar = torch.cumprod(1.0 - beta, dim=0)

    def _make_t(self, t, bs):
        if isinstance(t, int):
            t = torch.full((bs,), t, device=self.device, dtype=torch.long)
        else:
            t = t.to(self.device).long()
        return t.clamp(0, T - 1)

    @torch.no_grad()
    def denoise(self, x, t, strength=1.0, clamp=True):
        x = x.to(self.device).float()
        t_tensor = self._make_t(t, x.shape[0])
        eps = self.model(x, t_tensor)
        ab = self.alpha_bar[t_tensor][:, None, None, None]
        x0 = (x - torch.sqrt(1.0 - ab) * eps) / (torch.sqrt(ab) + 1e-8)
        if strength < 1.0:
            x0 = x + float(strength) * (x0 - x)
        if clamp:
            x0 = x0.clamp(CLAMP_MIN, CLAMP_MAX)
        return x0


# ============================================================
# LORA TRAINING
# ============================================================

def train_lora():
    print(f"\n{'=' * 60}\nLORA TRAINING V2.7 (epochs={LORA_EPOCHS}, "
          f"steps={LORA_STEPS_PER_EPOCH})\n{'=' * 60}")

    if not SCORE_MODEL_PATH.exists():
        print(f"[WARN] Score model not found: {SCORE_MODEL_PATH}")
        return False, []

    ds_train = CTImageDataset(TRAIN_DIR, augment=True,
                               max_images=LORA_MAX_IMAGES, seed=SEED)
    try:
        ds_val = CTImageDataset(VAL_DIR, augment=False,
                                 max_images=LORA_VAL_MAX, seed=SEED + 1)
    except Exception:
        ds_val = None

    samples_per_epoch = LORA_STEPS_PER_EPOCH * LORA_BATCH
    train_sampler = RandomSampler(ds_train, replacement=True,
                                   num_samples=samples_per_epoch)
    dl_train = DataLoader(ds_train, batch_size=LORA_BATCH, sampler=train_sampler,
                           num_workers=0, drop_last=True)

    dl_val = None
    if ds_val is not None:
        dl_val = DataLoader(ds_val, batch_size=LORA_BATCH, shuffle=False,
                             num_workers=0, drop_last=False)

    model = load_score_model(SCORE_MODEL_PATH, DEVICE, T)
    model = inject_lora(model, rank=LORA_RANK, alpha=LORA_ALPHA)

    for n, p in model.named_parameters():
        p.requires_grad = "lora_" in n

    lp = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(lp, lr=LORA_LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=LORA_EPOCHS,
                                                        eta_min=1e-7)
    crit = nn.MSELoss()

    best_val = float("inf")
    losses = []
    patience_count = 0
    t0 = time.time()

    for ep in range(LORA_EPOCHS):
        model.train()
        total = 0.0; nb = 0

        for x0 in tqdm(dl_train, desc=f"LoRA Ep{ep + 1}/{LORA_EPOCHS}",
                        disable=not HAS_TQDM):
            x0 = x0.to(DEVICE)
            t = torch.randint(0, T, (x0.shape[0],), device=DEVICE)
            xt, noise = ddpm_noise_image(x0, t)
            pred = model(xt, t)
            loss = crit(pred, noise)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(lp, 1.0)
            opt.step()
            total += loss.item(); nb += 1

        sched.step()
        avg_train = total / max(1, nb)

        val_loss = avg_train
        if dl_val is not None:
            model.eval()
            v_total = 0.0; v_nb = 0
            with torch.no_grad():
                for bi, xv in enumerate(dl_val):
                    if bi >= LORA_VAL_STEPS:
                        break
                    xv = xv.to(DEVICE)
                    tv = torch.randint(0, T, (xv.shape[0],), device=DEVICE)
                    xtv, nv = ddpm_noise_image(xv, tv)
                    pv = model(xtv, tv)
                    v_total += crit(pv, nv).item(); v_nb += 1
            val_loss = v_total / max(1, v_nb)

        losses.append(val_loss)

        elapsed = (time.time() - t0) / 60
        eta = elapsed / (ep + 1) * (LORA_EPOCHS - ep - 1)

        star = ""
        if val_loss < best_val:
            best_val = val_loss
            save_lora_weights(model, LORA_MODEL_PATH)
            star = " ★ (saved)"
            patience_count = 0
        else:
            patience_count += 1

        print(f"  Ep {ep + 1:02d}/{LORA_EPOCHS} | Train: {avg_train:.6f} | "
              f"Val: {val_loss:.6f} | Best: {best_val:.6f}{star} | ETA: {eta:.1f} min")

        if patience_count >= LORA_EARLY_STOP_PATIENCE:
            print(f"  [EARLY STOP] No improvement for {patience_count} epochs")
            break

    print(f"[OK] LoRA trained in {(time.time() - t0) / 60:.1f} min")
    return True, losses


# ============================================================
# HQS-ADMM SOLVER
# ============================================================

def geometric_schedule(s, e, k, K):
    if K <= 1:
        return float(e)
    return float(s) * ((float(e) / float(s)) ** (k / max(K - 1, 1)))


def linear_int_schedule(s, e, k, K):
    if K <= 1:
        return int(e)
    return int(round(s + (e - s) * k / max(K - 1, 1)))


@torch.no_grad()
def data_prox_nesterov(y, A, v, rho, n_iters, x0=None, ata_norm=None, weights=None):
    y = A.format_y(y); v = A.format_x(v)
    weights = format_weights(weights, A=A, y=y)
    if ata_norm is None:
        ata_norm = A.estimate_ata_norm(POWER_ITERS)
    w_lip = 1.0 if weights is None else max(float(weights.max().item()), 1.0)
    L = float(ata_norm) * w_lip + float(rho)
    step = 0.99 / (L + 1e-8)
    x = v.clone() if x0 is None else A.format_x(x0).clone()
    z = x.clone()
    t_acc = 1.0
    for _ in range(int(n_iters)):
        r = A.forward(z) - y
        if weights is not None:
            r = weights * r
        grad = A.at(r) + float(rho) * (z - v)
        x_new = (z - step * grad).clamp(CLAMP_MIN, CLAMP_MAX)
        t_new = 0.5 * (1.0 + math.sqrt(1.0 + 4.0 * t_acc * t_acc))
        beta = (t_acc - 1.0) / (t_new + 1e-12)
        z_new = x_new + beta * (x_new - x)
        restart_val = torch.sum((z - x_new) * (x_new - x)).item()
        if restart_val > 0:
            z = x_new.clone()
            t_acc = 1.0
        else:
            z = z_new.clamp(CLAMP_MIN, CLAMP_MAX)
            t_acc = t_new
        x = x_new
    return x.clamp(CLAMP_MIN, CLAMP_MAX)


@torch.no_grad()
def hqs_admm_reconstruct(y, A, prior, x_init=None, n_iters=ADMM_ITERS,
                          prox_iters=DATA_PROX_ITERS, use_prior=True,
                          verbose=False, weights=None):
    y = A.format_y(y)
    weights = format_weights(weights, A=A, y=y)
    x = A.fbp(y) if x_init is None else A.format_x(x_init).clone()
    z = x.clone()
    u = torch.zeros_like(x)
    ata_norm = A.estimate_ata_norm(POWER_ITERS)
    it = trange(n_iters, desc="ADMM") if (verbose and HAS_TQDM) else range(n_iters)
    for k in it:
        rho = geometric_schedule(RHO_START, RHO_END, k, n_iters)
        t = linear_int_schedule(DENOISE_T_START, DENOISE_T_END, k, n_iters)
        v = (z - u) if USE_ADMM else z
        x = data_prox_nesterov(y=y, A=A, v=v, rho=rho, n_iters=prox_iters,
                                x0=x, ata_norm=ata_norm, weights=weights)
        q = (x + u) if USE_ADMM else x
        if use_prior and prior is not None:
            z_new = prior.denoise(q, t=t, strength=PRIOR_STRENGTH, clamp=True)
        else:
            z_new = q.clamp(CLAMP_MIN, CLAMP_MAX)
        z = (1.0 - ADMM_RELAX) * z + ADMM_RELAX * z_new
        if USE_ADMM:
            u = u + x - z
    return z.clamp(CLAMP_MIN, CLAMP_MAX)


# ============================================================
# FIXED POINT V2.7 (ANCHORED GRADIENT + MOMENTUM + TRUST REGION)
# ============================================================

@torch.no_grad()
def data_fidelity_loss(x, y, A, weights=None):
    """Weighted data fidelity (blind)."""
    r = A.forward(x) - y
    if weights is not None:
        return float((weights * r * r).mean().item())
    return float((r * r).mean().item())


@torch.no_grad()
def _project_trust_region(x, x_anchor, radius):
    """Projects x into an L∞ ball of radius `radius` around x_anchor."""
    delta = (x - x_anchor).clamp(-radius, radius)
    return (x_anchor + delta).clamp(CLAMP_MIN, CLAMP_MAX)


@torch.no_grad()
def fixed_point_refinement(x, y, A, prior=None, n_iters=FP_ITERS, weights=None):
    """
    Fixed Point V2.7: anchored gradient descent (Halpern proximal) on
        min_x  0.5*||W^{1/2}(Ax - y)||^2 + β*||x - x_anchor||^2
    with Nesterov momentum and L∞ projection (trust region).

    - The diffusion prior is NOT used (avoids the destructive bias).
    - Line search with backtracking: the step is accepted only if the
      fidelity loss decreases by at least FP_MIN_REL_IMPROVE in relative terms.
    - L∞ projection: ||x - x_anchor||_∞ ≤ FP_TRUST_RADIUS → no drift.
    - Sanity check: if the final loss is not ≤ loss(x_lora), we
      return x_lora (absolute guarantee of not degrading).
    """
    x_anchor = A.format_x(x).clone()   # HQS+DDPM+LoRA output
    y = A.format_y(y)
    weights = format_weights(weights, A=A, y=y)

    # Reference
    ref_loss = data_fidelity_loss(x_anchor, y, A, weights)
    best_x = x_anchor.clone()
    best_loss = ref_loss

    # Base step via Lipschitz constant
    ata_norm = A.estimate_ata_norm(POWER_ITERS)
    w_lip = 1.0 if weights is None else max(float(weights.max().item()), 1.0)
    L = float(ata_norm) * w_lip + 2.0 * float(FP_ANCHOR_BETA)
    base_step = FP_STEP_INIT / (L + 1e-8)

    x = x_anchor.clone()
    prev_x = None

    for it in range(int(n_iters)):
        # Momentum point (Nesterov-like)
        if FP_USE_MOMENTUM and it > 0 and prev_x is not None:
            z = x + float(FP_MOMENTUM) * (x - prev_x)
            z = z.clamp(CLAMP_MIN, CLAMP_MAX)
        else:
            z = x.clone()

        # Gradient of the objective function
        r = A.forward(z) - y
        if weights is not None:
            r = weights * r
        grad = A.at(r) + 2.0 * float(FP_ANCHOR_BETA) * (z - x_anchor)

        # Line search with backtracking
        step = base_step
        accepted = False
        for bt in range(int(FP_MAX_BACKTRACK)):
            x_new = (z - step * grad).clamp(CLAMP_MIN, CLAMP_MAX)
            # Projection into the trust region
            x_new = _project_trust_region(x_new, x_anchor, FP_TRUST_RADIUS)
            new_loss = data_fidelity_loss(x_new, y, A, weights)

            if new_loss < best_loss * (1.0 - float(FP_MIN_REL_IMPROVE)):
                best_loss = new_loss
                best_x = x_new.clone()
                prev_x = x.clone()
                x = x_new
                accepted = True
                break
            else:
                step *= float(FP_BACKTRACK_FACTOR)

        if not accepted:
            break   # convergence reached

    # Absolute safety
    final_loss = data_fidelity_loss(best_x, y, A, weights)
    if final_loss > ref_loss:
        return x_anchor.clamp(CLAMP_MIN, CLAMP_MAX)
    return best_x.clamp(CLAMP_MIN, CLAMP_MAX)


# ============================================================
# UQ CALIBRATION
# ============================================================

def _conformal_quantile_1d(scores, alpha=0.05):
    scores = scores.flatten().float()
    n = int(scores.numel())
    if n <= 1:
        return 2.2
    q_level = math.ceil((n + 1) * (1.0 - alpha)) / n
    q_level = min(max(q_level, 0.0), 1.0)
    return float(torch.quantile(scores, q_level).item())


def calibrate_uq(A, prior, n_calib=UQ_CALIB_N_VAL, alpha=UQ_CALIB_ALPHA):
    print(f"\n[CALIB] Conformal UQ calibration on {n_calib} volumes...")
    try:
        vds = CTImageDataset(VAL_DIR, augment=False, max_images=n_calib, seed=SEED + 99)
    except Exception:
        return 2.2

    all_scores = []
    n_mc = max(4 if DEBUG_MODE else 8, min(int(UQ_SAMPLES), 16))

    for idx in range(min(n_calib, len(vds))):
        xgt = vds[idx].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            yc = A.forward(xgt)
            yn, det = simulate_ct_measurement(yc, seed=SEED + 500 + idx)
            w = estimate_data_weights_from_counts(det)

        mean = None; M2 = None; n = 0
        for s in range(n_mc):
            torch.manual_seed(SEED + 2000 + idx * 100 + s)
            yp = sample_sinogram_for_uq(yn, weights=w, scale=UQ_NOISE_SCALE)
            x0 = A.fbp(yp)
            xa = hqs_admm_reconstruct(y=yp, A=A, prior=prior, x_init=x0,
                                       n_iters=UQ_ADMM_ITERS,
                                       prox_iters=UQ_DATA_PROX_ITERS,
                                       use_prior=True, weights=w)
            xf = fixed_point_refinement(x=xa, y=yp, A=A, prior=prior,
                                         n_iters=UQ_FP_ITERS,
                                         weights=w).detach().cpu()
            n += 1
            if mean is None:
                mean = xf.clone(); M2 = torch.zeros_like(xf)
            else:
                delta = xf - mean; mean = mean + delta / n
                M2 = M2 + delta * (xf - mean)
            del yp, x0, xa, xf
            free_mem()

        if mean is None or n < 2:
            continue
        var = M2 / max(n - 1, 1)
        std = torch.sqrt(var + UQ_MIN_STD ** 2)
        ratio = (xgt.detach().cpu() - mean).abs() / (std + 1e-6)
        flat = ratio.flatten()
        if flat.numel() > UQ_CALIB_PIXELS_PER_IMAGE:
            g = torch.Generator(); g.manual_seed(SEED + 9000 + idx)
            ids = torch.randperm(flat.numel(), generator=g)[:UQ_CALIB_PIXELS_PER_IMAGE]
            flat = flat[ids]
        all_scores.append(flat.float().cpu())
        sc90 = float(torch.quantile(flat, 0.90).item())
        sc95 = float(torch.quantile(flat, 0.95).item())
        print(f"  val {idx} score q90={sc90:.3f} | q95={sc95:.3f}")
        del xgt, yc, yn, det, w, mean, M2, var, std, ratio, flat
        A.clean_astra_tables(); free_mem()

    if not all_scores:
        return 2.2
    scores = torch.cat(all_scores).float()
    cf_raw = _conformal_quantile_1d(scores, alpha=alpha)
    cf = max(UQ_MIN_CF, min(cf_raw, UQ_MAX_CF))
    if cf_raw > UQ_MAX_CF:
        print(f"[CALIB-WARN] Raw factor {cf_raw:.3f} capped at {UQ_MAX_CF:.1f}")
    print(f"[CALIB] Retained calibrated factor: {cf:.3f}")
    return cf


_UQ_CF = 2.2


@torch.no_grad()
def posterior_uq_streaming(y, rec_fn, n_samples=UQ_SAMPLES, gt=None, weights=None):
    global _UQ_CF
    if isinstance(y, np.ndarray):
        yd = torch.from_numpy(np.asarray(y, np.float32)).to(DEVICE)
    else:
        yd = y.detach().float().to(DEVICE)
    weights = format_weights(weights, A=None, y=yd)

    mean = None; M2 = None; n = 0
    for s in range(int(n_samples)):
        sv = SEED + 1000 + s
        torch.manual_seed(sv)
        yp = sample_sinogram_for_uq(yd, weights=weights, scale=UQ_NOISE_SCALE)
        try:
            xs = rec_fn(yp, seed=sv).detach().cpu()
        except Exception as e:
            print(f"  [UQ-SKIP] Sample {s} skipped: {e}")
            del yp; free_mem(); continue
        n += 1
        if mean is None:
            mean = xs.clone(); M2 = torch.zeros_like(xs)
        else:
            delta = xs - mean; mean = mean + delta / n
            M2 = M2 + delta * (xs - mean)
        del xs, yp; free_mem()

    if mean is None:
        return {"mean": torch.zeros(1, 1, IMG_SIZE, IMG_SIZE),
                "std": torch.ones(1, 1, IMG_SIZE, IMG_SIZE) * 1e-3,
                "ci95_low": torch.zeros(1, 1, IMG_SIZE, IMG_SIZE),
                "ci95_high": torch.ones(1, 1, IMG_SIZE, IMG_SIZE),
                "coverage_95": 0.0, "mean_ci_width": 1.0, "calib_factor": _UQ_CF}

    var = M2 / max(n - 1, 1) if n > 1 else torch.zeros_like(mean)
    std = torch.sqrt(var + UQ_MIN_STD ** 2)
    cf = float(_UQ_CF)
    cil = (mean - cf * std).clamp(0.0, 1.0)
    cih = (mean + cf * std).clamp(0.0, 1.0)
    r = {"mean": mean, "std": std, "ci95_low": cil, "ci95_high": cih, "calib_factor": cf}
    if gt is not None:
        gtc = gt.detach().cpu()
        r["coverage_95"] = float(((gtc >= cil) & (gtc <= cih)).float().mean())
        r["mean_ci_width"] = float((cih - cil).mean())
    else:
        r["coverage_95"] = 0.0
        r["mean_ci_width"] = float((cih - cil).mean())
    return r


# ============================================================
# GRID SEARCH
# ============================================================

def grid_search_prior_strength(A, prior, n_val=GRID_N_VAL):
    global PRIOR_STRENGTH
    print(f"\n[GRID SEARCH] Optimizing PRIOR_STRENGTH on {n_val} vals...")
    try:
        vds = CTImageDataset(VAL_DIR, augment=False, max_images=n_val, seed=SEED + 42)
    except Exception:
        return PRIOR_STRENGTH

    candidates = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
    old_ps = PRIOR_STRENGTH
    best_s = PRIOR_STRENGTH
    best_score = -999.0

    for strength in candidates:
        PRIOR_STRENGTH = strength
        psnrs = []; ssims = []
        for idx in range(min(n_val, len(vds))):
            xgt = vds[idx].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                yc = A.forward(xgt)
                yn, det = simulate_ct_measurement(yc, seed=SEED + 100 + idx)
                w = estimate_data_weights_from_counts(det)
            xfbp = A.fbp(yn)
            xrec = hqs_admm_reconstruct(y=yn, A=A, prior=prior, x_init=xfbp,
                                         n_iters=min(ADMM_ITERS, 18),
                                         prox_iters=min(DATA_PROX_ITERS, 12),
                                         use_prior=True, weights=w)
            xrec = fixed_point_refinement(x=xrec, y=yn, A=A, prior=prior,
                                           n_iters=min(FP_ITERS, 6), weights=w)
            psnrs.append(compute_psnr(xrec, xgt))
            ssims.append(compute_ssim(xrec, xgt))
            del xgt, yc, yn, det, w, xfbp, xrec
            free_mem()
        A.clean_astra_tables()
        avg_p = float(np.mean(psnrs)) if psnrs else -999.0
        avg_s = float(np.mean(ssims)) if ssims else 0.0
        score = avg_p + 30.0 * avg_s
        print(f"  strength={strength:.2f} -> PSNR={avg_p:.2f} | "
              f"SSIM={avg_s:.4f} | score={score:.2f}")
        if score > best_score:
            best_score = score; best_s = strength
    PRIOR_STRENGTH = old_ps
    print(f"[GRID] Optimal: {best_s:.2f}")
    return best_s


# ============================================================
# FIGURE SAVING (now includes clean and noisy sinograms)
# ============================================================

def _save_example_isolated(idx, xgt, yc, yn, xfbp, xprop, uq_std, metrics, fdir):
    try:
        gt = to_numpy_img(xgt)
        sino_clean = to_numpy_raw(yc)
        sino_noisy = to_numpy_raw(yn)
        fbp = to_numpy_img(xfbp)
        prop = to_numpy_img(xprop)
        err = np.abs(prop - gt)
        std = to_numpy_img(uq_std)

        fig, axes = plt.subplots(1, 7, figsize=(26, 3.8))
        imgs = [gt, sino_clean, sino_noisy, fbp, prop, err, std]
        titles = ["Original (GT)", "Clean Sinogram", "Noisy Sinogram",
                  "FBP", "CT-Diffusion-UQ", "Abs Error", "Uncertainty (Std)"]
        cmaps = ["gray", "gray", "gray", "gray", "gray", "hot", "magma"]
        vmaxs = [1.0, None, None, 1.0, 1.0, None, None]

        for ax, img, title, cmap, vmax in zip(axes, imgs, titles, cmaps, vmaxs):
            im = ax.imshow(img, cmap=cmap, vmin=0.0, vmax=vmax)
            ax.set_title(title, fontsize=10, fontweight="bold")
            ax.axis("off")
            if cmap not in ("gray",):
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        pfbp = metrics["fbp"]["psnr"][-1]
        pprop = metrics["hqs_ddpm_lora_fp"]["psnr"][-1]
        axes[0].text(-20, IMG_SIZE // 2,
                     f"#{idx:02d}\nFBP: {pfbp:.1f}dB\nProp: {pprop:.1f}dB",
                     fontsize=9, va="center", ha="right", weight="bold")
        plt.subplots_adjust(wspace=0.3)
        plt.savefig(fdir / f"example_{idx:03d}.png", dpi=130, bbox_inches="tight")
        plt.close(fig)
    except Exception as e:
        print(f"  [WARN] Failed to save figure #{idx}: {e}")
        try: plt.close("all")
        except Exception: pass


# ============================================================
# FULL EVALUATION
# ============================================================

def evaluate_full(A, prior_base, prior_lora, n_images=N_TEST_IMAGES,
                   save_n=N_EXAMPLES_TO_SAVE):
    print(f"\n{'=' * 70}\nEVALUATION  ({NUM_ANGLES} angles, NO ORACLE)\n{'=' * 70}")
    test_ds = CTImageDataset(TEST_DIR, augment=False)
    n_eval = min(n_images, len(test_ds)) if n_images else len(test_ds)
    print(f"[OK] Test images: {len(test_ds)} | Evaluation: {n_eval}")

    methods = ["fbp", "hqs_only", "hqs_ddpm", "hqs_ddpm_lora", "hqs_ddpm_lora_fp"]
    metrics = {m: {"psnr": [], "ssim": [], "rmse": [], "mae": [], "nrmse": []}
               for m in methods}
    metrics["coverage_95"] = []
    metrics["ci_width"] = []
    roi_metrics = {n: {"psnr": [], "ssim": [], "rmse": [], "mae": []}
                   for n in ROI_DEFINITIONS}

    csv_path = OUT_DIR / "test_results.csv"
    csv_f = open(csv_path, "w", newline="")
    csv_w = csv.writer(csv_f)
    csv_w.writerow(["idx", "file"]
                   + [f"psnr_{m}" for m in methods]
                   + [f"ssim_{m}" for m in methods]
                   + ["cov", "ciw"])
    t0 = time.time()

    for idx in range(n_eval):
        try:
            xgt = test_ds[idx].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                yc = A.forward(xgt)
                yn, det = simulate_ct_measurement(yc, seed=SEED + idx)
                w = estimate_data_weights_from_counts(det)

            xfbp = A.fbp(yn)
            x_hqs = hqs_admm_reconstruct(y=yn, A=A, prior=None, x_init=xfbp,
                                          use_prior=False, weights=w)
            x_ddpm = hqs_admm_reconstruct(y=yn, A=A, prior=prior_base, x_init=xfbp,
                                           use_prior=True, weights=w)
            x_lora = hqs_admm_reconstruct(y=yn, A=A, prior=prior_lora, x_init=xfbp,
                                           use_prior=True, weights=w)
            # Fixed point V2.7: ANCHORED GRADIENT + MOMENTUM + TRUST REGION
            x_proposed = fixed_point_refinement(x=x_lora, y=yn, A=A, prior=prior_lora,
                                                 n_iters=FP_ITERS, weights=w)

            recs = {"fbp": xfbp, "hqs_only": x_hqs, "hqs_ddpm": x_ddpm,
                    "hqs_ddpm_lora": x_lora, "hqs_ddpm_lora_fp": x_proposed}
            for mn, xr in recs.items():
                m = compute_all_metrics(xr, xgt)
                for k, v in m.items():
                    metrics[mn][k].append(v)
            roi = compute_roi_metrics(x_proposed, xgt)
            for name, vals in roi.items():
                for k, v in vals.items():
                    roi_metrics[name][k].append(v)

            def rec_fast(yi, seed=None):
                if seed is not None: seed_everything(seed)
                x0 = A.fbp(yi)
                xa = hqs_admm_reconstruct(y=yi, A=A, prior=prior_lora, x_init=x0,
                                           n_iters=UQ_ADMM_ITERS,
                                           prox_iters=UQ_DATA_PROX_ITERS,
                                           use_prior=True, weights=w)
                # Fixed Point V2.7 for UQ samples
                xf = fixed_point_refinement(x=xa, y=yi, A=A, prior=prior_lora,
                                             n_iters=UQ_FP_ITERS, weights=w)
                return xf

            uq = posterior_uq_streaming(y=yn, rec_fn=rec_fast,
                                         n_samples=UQ_SAMPLES, gt=xgt, weights=w)
            cov = uq.get("coverage_95", 0.0)
            ciw = uq.get("mean_ci_width", 0.0)
            metrics["coverage_95"].append(cov)
            metrics["ci_width"].append(ciw)

            if idx < save_n:
                _save_example_isolated(idx, xgt, yc, yn, xfbp, x_proposed, uq["std"],
                                        metrics, FIGURES_DIR)

            csv_w.writerow([idx, test_ds.files[idx].name]
                           + [f"{metrics[m]['psnr'][-1]:.2f}" for m in methods]
                           + [f"{metrics[m]['ssim'][-1]:.4f}" for m in methods]
                           + [f"{cov:.4f}", f"{ciw:.6f}"])
            csv_f.flush()
            A.clean_astra_tables()
            del xfbp, x_hqs, x_ddpm, x_lora, x_proposed
            del xgt, yc, yn, det, w, uq
            free_mem()

            if (idx + 1) % BATCH_LOG_EVERY == 0 or idx < 3:
                ela = time.time() - t0
                eta = ela / (idx + 1) * (n_eval - idx - 1)
                gain_lora_fp = (np.mean(metrics['hqs_ddpm_lora_fp']['psnr'])
                                 - np.mean(metrics['hqs_ddpm_lora']['psnr']))
                print(f"  [{idx + 1:02d}/{n_eval:02d}] "
                      f"FBP: {np.mean(metrics['fbp']['psnr']):.2f} | "
                      f"DDPM: {np.mean(metrics['hqs_ddpm']['psnr']):.2f} | "
                      f"LoRA: {np.mean(metrics['hqs_ddpm_lora']['psnr']):.2f} | "
                      f"Proposed: {np.mean(metrics['hqs_ddpm_lora_fp']['psnr']):.2f} dB "
                      f"[Δ FP: {gain_lora_fp:+.3f}] "
                      f"SSIM: {np.mean(metrics['hqs_ddpm_lora_fp']['ssim']):.4f} "
                      f"| ETA: {eta / 60:.1f} min")
        except Exception as e:
            print(f"[ERROR image {idx}]: {e}")
            A.clean_astra_tables(); free_mem()
            continue

    csv_f.close()
    print(f"\n[OK] Evaluation completed in {(time.time() - t0) / 60:.1f} min")
    return metrics, roi_metrics


# ============================================================
# RESULTS
# ============================================================

def print_full_results(metrics, roi_metrics):
    print(f"\n{'=' * 82}\nCOMPLETE ABLATION TABLE V2.7\n{'=' * 82}")
    print(f"{'Method':<26}{'PSNR (Mean±Std)':>18}{'Median':>10}"
          f"{'SSIM (Mean±Std)':>18}{'Median':>10}")
    print("-" * 82)
    labels = {
        "fbp": "1. Standard FBP",
        "hqs_only": "2. HQS (no diffusion)",
        "hqs_ddpm": "3. HQS + DDPM",
        "hqs_ddpm_lora": "4. HQS + DDPM + LoRA",
        "hqs_ddpm_lora_fp": "5. CT-Diffusion-UQ (+ FP)",
    }
    ordered = ["fbp", "hqs_only", "hqs_ddpm", "hqs_ddpm_lora", "hqs_ddpm_lora_fp"]
    for k in ordered:
        m = metrics[k]
        if not m["psnr"]:
            continue
        pm = np.mean(m["psnr"]); ps = np.std(m["psnr"]); pmed = np.median(m["psnr"])
        sm = np.mean(m["ssim"]); ss = np.std(m["ssim"]); smed = np.median(m["ssim"])
        print(f"{labels[k]:<26}{pm:>8.2f} ± {ps:.2f}{pmed:>10.2f}"
              f"{sm:>9.4f} ± {ss:.4f}{smed:>10.4f}")
    print("=" * 82)
    if metrics["fbp"]["psnr"] and metrics["hqs_ddpm_lora_fp"]["psnr"]:
        base = np.array(metrics["fbp"]["psnr"])
        prop = np.array(metrics["hqs_ddpm_lora_fp"]["psnr"])
        n = min(len(base), len(prop))
        gain = prop[:n] - base[:n]
        print(f"Total gain CT-Diffusion-UQ vs FBP: ΔPSNR = {np.mean(gain):+.2f} dB")
    if metrics["hqs_ddpm_lora"]["psnr"] and metrics["hqs_ddpm_lora_fp"]["psnr"]:
        lora = np.array(metrics["hqs_ddpm_lora"]["psnr"])
        prop = np.array(metrics["hqs_ddpm_lora_fp"]["psnr"])
        n = min(len(lora), len(prop))
        gain_fp = prop[:n] - lora[:n]
        print(f"Gain FP  vs LoRA alone: ΔPSNR = {np.mean(gain_fp):+.3f} dB")

    try:
        from scipy.stats import wilcoxon
        print(f"\n{'=' * 60}\nSIGNIFICANCE TEST (Wilcoxon)\n{'=' * 60}")
        prop = np.array(metrics["hqs_ddpm_lora_fp"]["psnr"])
        for k in ["fbp", "hqs_only", "hqs_ddpm", "hqs_ddpm_lora"]:
            ref = np.array(metrics[k]["psnr"])
            n = min(len(prop), len(ref))
            if n < 2:
                continue
            try:
                stat, pval = wilcoxon(prop[:n], ref[:n], alternative="greater")
            except Exception:
                continue
            sig = ("***" if pval < 0.001 else "**" if pval < 0.01
                   else "*" if pval < 0.05 else "ns")
            print(f"  CT-Diffusion-UQ vs {labels[k]:<24} : p = {pval:.4e} ({sig})")
    except Exception:
        pass

    if metrics["coverage_95"]:
        print(f"\nUQ: 95% Coverage = {np.mean(metrics['coverage_95']):.4f}")
        print(f"Mean width = {np.mean(metrics['ci_width']):.4f}")


def save_summary_file(metrics, path):
    with open(path, "w", encoding="utf-8") as f:
        f.write("=== REPORT V2.7 ===\n\n")
        labels = {"fbp": "FBP", "hqs_only": "HQS", "hqs_ddpm": "+DDPM",
                  "hqs_ddpm_lora": "+LoRA", "hqs_ddpm_lora_fp": "CT-Diffusion-UQ"}
        for k in labels:
            m = metrics[k]
            if not m["psnr"]:
                continue
            f.write(f"{labels[k]:<18} : PSNR = {np.mean(m['psnr']):.2f} ± "
                    f"{np.std(m['psnr']):.2f} dB | SSIM = {np.mean(m['ssim']):.4f} ± "
                    f"{np.std(m['ssim']):.4f}\n")
        if metrics["coverage_95"]:
            f.write(f"\n95% Coverage: {np.mean(metrics['coverage_95']):.4f}\n")
        f.write(f"Calibration factor: {_UQ_CF:.3f}\n")


def plot_ablation_figure(metrics):
    keys = ["fbp", "hqs_only", "hqs_ddpm", "hqs_ddpm_lora", "hqs_ddpm_lora_fp"]
    labels = ["FBP", "HQS", "+DDPM", "+LoRA", "CT-Diffusion-UQ"]
    if any(len(metrics[k]["psnr"]) == 0 for k in keys):
        return
    psnrs = [np.mean(metrics[k]["psnr"]) for k in keys]
    ssims = [np.mean(metrics[k]["ssim"]) for k in keys]
    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(labels)); w = 0.35
    b1 = ax1.bar(x - w / 2, psnrs, w, color="#2b5c8f", edgecolor="k", label="PSNR")
    ax1.set_ylabel("PSNR (dB)", color="#2b5c8f", fontweight="bold")
    ax1.set_xticks(x); ax1.set_xticklabels(labels, fontweight="bold")
    ax1.grid(axis="y", linestyle="--", alpha=0.5)
    ax2 = ax1.twinx()
    b2 = ax2.bar(x + w / 2, ssims, w, color="#d95f02", edgecolor="k", label="SSIM")
    ax2.set_ylabel("SSIM", color="#d95f02", fontweight="bold")
    ax2.set_ylim(0.0, 1.0)
    for bar, val in zip(b1, psnrs):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f"{val:.1f}", ha="center", fontsize=8, weight="bold")
    for bar, val in zip(b2, ssims):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 f"{val:.3f}", ha="center", fontsize=8, weight="bold")
    plt.title("Ablation V2.7", fontweight="bold", fontsize=12)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig_ablation.png", dpi=150)
    plt.close(fig)


def plot_correlation_figure(metrics):
    fbp = metrics["fbp"]["psnr"]; prop = metrics["hqs_ddpm_lora_fp"]["psnr"]
    if not fbp or not prop:
        return
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(fbp, prop, color="#2b5c8f", edgecolors="k", s=60, alpha=0.8)
    mi = min(min(fbp), min(prop)) - 1; ma = max(max(fbp), max(prop)) + 1
    ax.plot([mi, ma], [mi, ma], "k--")
    ax.set_xlabel("FBP PSNR (dB)", fontweight="bold")
    ax.set_ylabel("CT-Diffusion-UQ PSNR (dB)", fontweight="bold")
    ax.set_title(f"Mean gain: +{np.mean(prop) - np.mean(fbp):.2f} dB",
                 fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "fig_correlation.png", dpi=150)
    plt.close(fig)


# ============================================================
# MAIN
# ============================================================

def main():
    global _UQ_CF, PRIOR_STRENGTH
    ensure_dirs(); seed_everything(SEED)
    t0 = time.time()
    mode = "DEBUG" if DEBUG_MODE else "FINAL"
    print("=" * 70)
    print(f"CT DIFFUSION  - PIPELINE ({mode})")
    print(f"Device: {DEVICE} | Backend: {CT_BACKEND}")
    print(f"Test: {N_TEST_IMAGES} | UQ: {UQ_SAMPLES}")
    print("=" * 70)

    prepare_split(seed=SEED, force=FORCE_RESPLIT)

    if not SCORE_MODEL_PATH.exists():
        print(f"[ERROR] Model not found: {SCORE_MODEL_PATH}")
        return

    if FORCE_LORA_RETRAIN or (not LORA_MODEL_PATH.exists()) or FORCE_RESPLIT:
        train_lora()
    else:
        print(f"[OK] Existing LoRA weights: {LORA_MODEL_PATH}")

    print("\n[INIT] CT Operator...")
    A = CTOperator(img_size=IMG_SIZE, num_angles=NUM_ANGLES, device=DEVICE,
                    angle_max_deg=ANGLE_MAX_DEG, backend=CT_BACKEND)
    print(f"  Active backend: {A.backend} | Detectors: {A.det_count}")
    print(f"  ||AᵀA||: {A.estimate_ata_norm(POWER_ITERS):.2f}")
    test_adjoint(A); test_phantom(A)

    model_base = load_score_model(SCORE_MODEL_PATH, DEVICE, T)
    prior_base = DiffusionScorePrior(model_base, DEVICE)
    print("  [OK] Base DDPM prior initialized")

    model_lora = load_score_model(SCORE_MODEL_PATH, DEVICE, T)
    model_lora = inject_lora(model_lora, rank=LORA_RANK, alpha=LORA_ALPHA)
    if LORA_MODEL_PATH.exists():
        load_lora_weights(model_lora, LORA_MODEL_PATH, device=DEVICE)
    model_lora.eval()
    prior_lora = DiffusionScorePrior(model_lora, DEVICE)
    print("  [OK] DDPM + LoRA prior initialized")

    if not DEBUG_MODE:
        PRIOR_STRENGTH = grid_search_prior_strength(A, prior_lora, n_val=GRID_N_VAL)
        print(f"[FINAL PARAM] PRIOR_STRENGTH = {PRIOR_STRENGTH}")

    _UQ_CF = calibrate_uq(A, prior_lora, n_calib=UQ_CALIB_N_VAL, alpha=UQ_CALIB_ALPHA)

    metrics, roi_metrics = evaluate_full(A, prior_base, prior_lora,
                                          n_images=N_TEST_IMAGES,
                                          save_n=N_EXAMPLES_TO_SAVE)

    A.clean_astra_tables()
    del prior_base, prior_lora, model_base, model_lora, A
    free_mem()

    print_full_results(metrics, roi_metrics)
    save_summary_file(metrics, OUT_DIR / "results_summary_v27.txt")

    psnr_final = np.mean(metrics["hqs_ddpm_lora_fp"]["psnr"]) if metrics["hqs_ddpm_lora_fp"]["psnr"] else 0.0
    ssim_final = np.mean(metrics["hqs_ddpm_lora_fp"]["ssim"]) if metrics["hqs_ddpm_lora_fp"]["ssim"] else 0.0
    cov_final = np.mean(metrics["coverage_95"]) if metrics["coverage_95"] else 0.0

    print(f"\n{'=' * 70}\nARTICLE Q1 OBJECTIVES\n{'=' * 70}")
    print(f"1. Final PSNR        : {psnr_final:.2f} dB  (CT-Diffusion-UQ vs LoRA alone: See Δ FP)")
    print(f"2. SSIM > 0.90       : {ssim_final:.4f}    "
          f"{'[VALIDATED]' if ssim_final >= 0.90 else '[NOT VALIDATED]'}")
    print(f"3. 95% CI Coverage   : {cov_final:.4f}    "
          f"{'[VALIDATED]' if abs(cov_final - 0.95) < 0.05 else '[NOT VALIDATED]'}")

    plot_ablation_figure(metrics)
    plot_correlation_figure(metrics)

    np.savez(OUT_DIR / "all_metrics_v27.npz",
             **{f"{m}_{k}": np.array(v) for m in metrics
                if m not in ("coverage_95", "ci_width")
                for k, v in metrics[m].items()},
             coverage_95=np.array(metrics["coverage_95"]),
             ci_width=np.array(metrics["ci_width"]),
             calib_factor=np.array([_UQ_CF]))

    print(f"\n[DONE] Total time: {(time.time() - t0) / 60:.1f} min")
    print(f"Results: {OUT_DIR.resolve()}")


if __name__ == "__main__":
    main()

CT DIFFUSION  - PIPELINE (FINAL)
Device: cpu | Backend: astra
Test: 30 | UQ: 10
[OK] Existing split validated: 60687|7585|7587
[OK] Existing LoRA weights: C:\Users\user\Desktop\hassan jrod\models\lora_v2.pth

[INIT] CT Operator...
  Active backend: astra_cpu | Detectors: 364
  ||AᵀA||: 44493.91

[ADJOINT] <Ax,y>=1.5014e+03 | <x,ATy>=1.5014e+03 | err=3.66e-07 [OK]
[PHANTOM] Sinogram: (1, 180, 364) | FBP PSNR=32.32 dB
  [OK] Base DDPM prior initialized
[OK] LoRA loaded (12 tensors)
  [OK] DDPM + LoRA prior initialized

[GRID SEARCH] Optimizing PRIOR_STRENGTH on 50 vals...
  strength=0.25 -> PSNR=31.89 | SSIM=0.9344 | score=59.92
  strength=0.30 -> PSNR=31.19 | SSIM=0.9341 | score=59.21
  strength=0.40 -> PSNR=29.92 | SSIM=0.9289 | score=57.78
  strength=0.45 -> PSNR=29.37 | SSIM=0.9251 | score=57.13
  strength=0.50 -> PSNR=28.89 | SSIM=0.9209 | score=56.52
[GRID] Optimal: 0.25
[FINAL PARAM] PRIOR_STRENGTH = 0.25

[CALIB] Conformal UQ calibration on 5 volumes...
  val 0 score q90=0.669 | 